# Agent Evaluation: Trajectories, Tool Use, and Task Completion

Evaluating an agent requires measuring the full trajectory, not just the final output. A correct answer via 40 unnecessary steps is a poorly-performing agent. A wrong answer via flawless reasoning is a differently-failing agent.

## The Agent Evaluation Stack

Agent evaluation has several layers:

**Task completion rate**: did the agent successfully complete the end goal? The primary metric.

**Tool use accuracy**: were the right tools called with correct arguments? Hallucinated tool calls (calling tools that don't exist) and malformed arguments are tracked separately.

**Step efficiency**: actual steps / minimum steps. Efficiency < 1 is impossible; efficiency > 2 indicates significant wasted computation.

**Error recovery rate**: when the agent encounters an error (tool failure, wrong result), what fraction of the time does it successfully recover?

**Safety compliance**: did the agent stay within its authorized scope? Did it attempt any unauthorized actions?

In [ ]:
from dataclasses import dataclass, field
from typing import Callable

@dataclass
class EvalCase:
    task: str
    expected_answer: str
    min_steps: int
    available_tools: set
    requires_tools: list

@dataclass
class TrajectoryEval:
    task: str
    steps: int
    tool_calls: list
    final_answer: str
    correct: bool
    efficiency: float
    hallucinated_tools: list
    tool_precision: float

def evaluate_trajectory(case: EvalCase, trajectory: list, final_answer: str) -> TrajectoryEval:
    tool_calls = [s for s in trajectory if s.get('type') == 'tool_call']
    n_steps = len(trajectory)
    efficiency = case.min_steps / max(1, n_steps)
    hallucinated = [t['tool'] for t in tool_calls if t['tool'] not in case.available_tools]
    precision = 1 - len(hallucinated) / max(1, len(tool_calls))
    required_used = sum(1 for req in case.requires_tools
                         if any(t['tool'] == req for t in tool_calls))
    correct = case.expected_answer.lower() in final_answer.lower()
    return TrajectoryEval(
        task=case.task[:50], steps=n_steps, tool_calls=tool_calls,
        final_answer=final_answer[:60], correct=correct,
        efficiency=round(efficiency, 2), hallucinated_tools=hallucinated,
        tool_precision=round(precision, 2)
    )

def aggregate_eval(evals: list) -> dict:
    n = len(evals)
    return {
        'n': n,
        'task_completion': sum(e.correct for e in evals) / n,
        'avg_efficiency': sum(e.efficiency for e in evals) / n,
        'avg_tool_precision': sum(e.tool_precision for e in evals) / n,
        'hallucination_rate': sum(len(e.hallucinated_tools) > 0 for e in evals) / n,
    }

cases = [
    EvalCase('What is 15% of 200?', '30', min_steps=2, available_tools={'calculator', 'search'}, requires_tools=['calculator']),
    EvalCase('Find the population of Tokyo', '13.96 million', min_steps=2, available_tools={'search'}, requires_tools=['search']),
]
trajectories = [
    [{'type': 'thought', 'content': 'Need to calculate 15% of 200'},
     {'type': 'tool_call', 'tool': 'calculator', 'args': {'expr': '200*0.15'}},
     {'type': 'answer', 'content': '30.0'}],
    [{'type': 'thought', 'content': 'Need Tokyo population'},
     {'type': 'tool_call', 'tool': 'google_search', 'args': {'q': 'Tokyo population'}},  # hallucinated tool
     {'type': 'answer', 'content': 'Around 13.96 million people live in Tokyo'}],
]
final_answers = ['The answer is 30.0', 'Around 13.96 million people live in Tokyo']

evals = [evaluate_trajectory(case, traj, ans)
         for case, traj, ans in zip(cases, trajectories, final_answers)]
for e in evals:
    print(f'Task: {e.task}')
    print(f'  Correct={e.correct}, Efficiency={e.efficiency}, Tool Precision={e.tool_precision}')
    if e.hallucinated_tools:
        print(f'  HALLUCINATED TOOLS: {e.hallucinated_tools}')
print('\nAggregate:')
for k, v in aggregate_eval(evals).items():
    print(f'  {k}: {v:.2f}' if isinstance(v, float) else f'  {k}: {v}')